# FOXF1_bead — 05_live_quantification

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 05 Live Quantification

## Future Development Note

Current DAPI masking and downstream quantification operate on the **fixed small-image files**.
A future improvement worth testing is to perform masking and/or quantification in **large-image space**, since the large fields can contain real cell-containing regions that are truncated in the small images.
This is especially relevant for peripheral / monolayer cells that may be cut off by the small FOV.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq
from scripts import live_fixed_alignment as lfa

## Paths And Parameters

In [ ]:
POSITION_MANIFEST = ROOT / "results/manifests/analysis_position_manifest.tsv"
LIVE_SMALL_CENTROIDS = ROOT / "results/annotations/well_centroids_small_mapped.tsv"
LIVE_FIXED_TSV = ROOT / "results/annotations/live_fixed_alignment_transforms.tsv"

OUT_DIR = ROOT / "results/measurements/live_fixed_small"
QC_DIR = ROOT / "results/qc/05_live_quantification"
MASK_DIR = ROOT / "results/masks/live_fixed_small"
FIXED_MASK_FINAL_DIR = MASK_DIR / "fixed_mask_final04_npz"
LIVE_MASK_FINAL_DIR = MASK_DIR / "live_mask_final04_npz"
for path in [OUT_DIR, QC_DIR, MASK_DIR, FIXED_MASK_FINAL_DIR, LIVE_MASK_FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LIVE_COHORT_ID = "2026-01-22_day2_live"
FIXED_COHORT_ID = "2026-01-22_day2_fix"
LIVE_YFP_KEYWORDS = ["tagyfp", "foxf1", "yfp"]
BIN_UM = 35.0
LIVE_FIELD_SIGMA_LOW_PX = 40.0
LIVE_FIELD_SIGMA_HIGH_PX = 80.0
LIVE_FIELD_MIN_BACKGROUND_PIXELS = 50000
LIVE_FIELD_MODE = "reflect"
LIVE_FIELD_BLEND_RULE = "linear_support_fraction"

MASK_TRANSFER_EXAMPLE_POSITIONS = ["1-1", "1-3", "2-1", "2-6", "3-3", "5-6"]
FLATFIELD_EXAMPLE_POSITIONS = ["1-1", "1-3", "2-1", "2-6", "5-6"]

pos_df = common.load_position_manifest(POSITION_MANIFEST)
live_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[LIVE_COHORT_ID],
    conditions=["live"],
).sort_values(["canonical_position"]).reset_index(drop=True)
fixed_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[FIXED_COHORT_ID],
    conditions=["fixed"],
).sort_values(["canonical_position"]).reset_index(drop=True)
align_df = pd.read_csv(LIVE_FIXED_TSV, sep="	")
align_df["canonical_position"] = align_df["canonical_position"].astype(str)
live_pos_df["canonical_position"] = live_pos_df["canonical_position"].astype(str)
fixed_pos_df["canonical_position"] = fixed_pos_df["canonical_position"].astype(str)

live_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in live_pos_df.itertuples(index=False)}
fixed_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in fixed_pos_df.itertuples(index=False)}
align_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in align_df.itertuples(index=False)}

live_cent_df = pd.read_csv(LIVE_SMALL_CENTROIDS, sep="	")
live_cent_df["canonical_position"] = live_cent_df["canonical_position"].astype(str)


def robust_rescale_from_values(image: np.ndarray, valid_mask: np.ndarray | None = None, q_low: float = 1.0, q_high: float = 99.0) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    if valid_mask is None:
        keep = np.isfinite(arr)
    else:
        keep = np.asarray(valid_mask, dtype=bool) & np.isfinite(arr)
    if not np.any(keep):
        return np.zeros_like(arr, dtype=np.float32)
    vals = arr[keep]
    lo = float(np.percentile(vals, q_low))
    hi = float(np.percentile(vals, q_high))
    if not np.isfinite(lo):
        lo = float(np.nanmin(vals))
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    out = (arr - lo) / max(hi - lo, float(lfq.EPS))
    out = np.clip(out, 0.0, 1.0)
    out[~np.isfinite(out)] = 0.0
    return out.astype(np.float32)


def load_live_bundle(canonical_position: str) -> dict:
    row = live_by_pos[str(canonical_position)]
    img = common.read_czi(ROOT / row["primary_analysis_file"])
    bf_idx = common.find_channel_index(img.channels, ["bright"])
    yfp_idx = common.find_channel_index(img.channels, LIVE_YFP_KEYWORDS)
    if bf_idx is None or yfp_idx is None:
        raise RuntimeError(f"Missing live BF or TagYFP channel for {canonical_position}; channels={img.channels}")
    return {
        "row": row,
        "image": img,
        "bf": img.channel_images[int(bf_idx)].astype(np.float32),
        "yfp_raw": img.channel_images[int(yfp_idx)].astype(np.float32),
    }


def load_fixed_context_bundle(canonical_position: str) -> dict:
    row = fixed_by_pos[str(canonical_position)]
    bundle = lfq.load_czi_bundle_with_z(ROOT / row["primary_analysis_file"])
    bf_idx = common.find_channel_index(bundle["channels"], ["bright"])
    dapi_idx = common.find_channel_index(bundle["channels"], ["dapi"])
    if bf_idx is None or dapi_idx is None:
        raise RuntimeError(f"Missing fixed BF or DAPI channel for {canonical_position}; channels={bundle['channels']}")
    return {
        "row": row,
        "bf_max": bundle["channel_max_cyx"][int(bf_idx)].astype(np.float32),
        "dapi_max": bundle["channel_max_cyx"][int(dapi_idx)].astype(np.float32),
    }


## Final DAPI Masks Transferred To Live

In [ ]:
final04_masks = lfq.build_final04_fixed_dapi_masks(
    position_manifest=POSITION_MANIFEST,
    cohort_id=FIXED_COHORT_ID,
    root=ROOT,
)
final04_mask_summary_df = final04_masks["summary_df"].copy()
final04_mask_summary_df.to_csv(OUT_DIR / "fixed_final04_mask_summary.tsv", sep="	", index=False)

mask_transfer = lfq.transfer_fixed_masks_to_live_space(
    position_manifest=POSITION_MANIFEST,
    live_fixed_alignment_path=LIVE_FIXED_TSV,
    fixed_mask_payloads=final04_masks["mask_payloads"],
    root=ROOT,
    live_cohort_id=LIVE_COHORT_ID,
    fixed_cohort_id=FIXED_COHORT_ID,
    write_fixed_mask_dir=FIXED_MASK_FINAL_DIR,
    write_live_mask_dir=LIVE_MASK_FINAL_DIR,
)
live_mask_summary_df = mask_transfer["summary_df"].copy()
fixed_masks_by_position = mask_transfer["fixed_masks_by_position"]
live_masks_by_position = mask_transfer["live_masks_by_position"]
live_mask_summary_df.to_csv(OUT_DIR / "live_mask_transfer_summary.tsv", sep="	", index=False)

print("Approved final04 fixed-mask summary:")
display(final04_mask_summary_df.head())
print()
print("Transferred live-mask summary:")
print(live_mask_summary_df["status"].value_counts(dropna=False).to_string())
display(live_mask_summary_df.head())
print()
print(
    f"Round-2 pooled threshold = {float(final04_masks['round2_threshold']):.2f} "
    f"(mu={float(final04_masks['round2_fit']['mu_bg_raw']):.2f}, sigma={float(final04_masks['round2_fit']['sigma_bg_raw']):.2f})"
)

example_positions = [p for p in MASK_TRANSFER_EXAMPLE_POSITIONS if p in live_masks_by_position]
if not example_positions:
    example_positions = live_mask_summary_df.loc[live_mask_summary_df["status"] == "ok", "canonical_position"].astype(str).head(5).tolist()

fig, axes = plt.subplots(len(example_positions), 4, figsize=(17.0, 3.9 * len(example_positions)), constrained_layout=True)
axes = np.atleast_2d(axes)
for row_idx, pos in enumerate(example_positions):
    fixed_ctx = load_fixed_context_bundle(pos)
    live_ctx = load_live_bundle(pos)
    fixed_mask = np.asarray(fixed_masks_by_position[pos], dtype=bool)
    live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
    cent_sub = live_cent_df[(live_cent_df["canonical_position"].astype(str) == str(pos)) & (live_cent_df["mapping_status"] == "ok") & (live_cent_df["annotation_status"] == "annotated") & (live_cent_df["inside_small_fov"].fillna(False).astype(bool))].copy()
    beads_xy = cent_sub[["centroid_small_x_px", "centroid_small_y_px"]].to_numpy(dtype=np.float32) if len(cent_sub) else np.zeros((0, 2), dtype=np.float32)
    align_row = align_by_pos[str(pos)]
    fixed_to_live = lfa.affine_matrix_from_row_prefix(align_row, "fixed_small_to_live_small")
    warped_fixed_dapi = lfa.warp_image_with_affine(
        image=fixed_ctx["dapi_max"],
        forward_mat=fixed_to_live,
        output_shape_yx=live_ctx["bf"].shape,
        order=1,
        cval=np.nan,
    )

    axes[row_idx, 0].imshow(common._robust_rescale(fixed_ctx["dapi_max"]), cmap="magma")
    lfq._mask_contour(axes[row_idx, 0], fixed_mask, color="cyan")
    axes[row_idx, 0].set_title(f"{pos} fixed DAPI max + final mask")

    axes[row_idx, 1].imshow(common._robust_rescale(live_ctx["bf"]), cmap="gray")
    lfq._mask_contour(axes[row_idx, 1], live_mask, color="cyan")
    axes[row_idx, 1].set_title("Live BF + transferred mask\nmagenta circles = bead-containing wells")

    axes[row_idx, 2].imshow(common._robust_rescale(live_ctx["yfp_raw"]), cmap="Greens")
    lfq._mask_contour(axes[row_idx, 2], live_mask, color="cyan")
    axes[row_idx, 2].set_title("Live TagYFP raw + transferred mask")

    axes[row_idx, 3].imshow(common._robust_rescale(live_ctx["bf"]), cmap="gray")
    axes[row_idx, 3].imshow(robust_rescale_from_values(warped_fixed_dapi, np.isfinite(warped_fixed_dapi)), cmap="magma", alpha=0.55)
    lfq._mask_contour(axes[row_idx, 3], live_mask, color="cyan")
    axes[row_idx, 3].set_title("Warped fixed DAPI in live space")

    if beads_xy.size:
        for ax in axes[row_idx, 1:]:
            ax.scatter(beads_xy[:, 0], beads_xy[:, 1], s=42, facecolors="none", edgecolors="magenta", linewidths=1.2, label="bead-containing wells in small FOV")
        handles, labels = axes[row_idx, 1].get_legend_handles_labels()
        if labels:
            axes[row_idx, 1].legend(handles[:1], labels[:1], frameon=False, fontsize=7, loc="lower left")

    for ax in axes[row_idx]:
        ax.set_xticks([])
        ax.set_yticks([])

fig.savefig(QC_DIR / "live_mask_transfer_examples.png", dpi=160, bbox_inches="tight")
plt.show()

## Live YFP Flat-Field Estimation From Off-Mask Pixels

> Note for later refinement: the global unsupported-region map currently contains a small island of apparent support that is likely not a trustworthy background region. In most scenes there is probably cyst signal there or nearby. We are leaving it unchanged for now because this depends sensitively on the upstream DAPI mask refinement, which we plan to revisit only at the very end.


In [ ]:
live_field_payload = lfq.estimate_live_masked_illumination_field(
    position_manifest=POSITION_MANIFEST,
    cohort_id=LIVE_COHORT_ID,
    live_masks_by_position=live_masks_by_position,
    channel_keywords=LIVE_YFP_KEYWORDS,
    root=ROOT,
    field_gaussian_sigma_px=LIVE_FIELD_SIGMA_LOW_PX,
    min_background_pixels=LIVE_FIELD_MIN_BACKGROUND_PIXELS,
    smoothing_mode=LIVE_FIELD_MODE,
)
live_field_position_df = live_field_payload["position_df"].copy()

masked_mean = np.asarray(live_field_payload["masked_mean"], dtype=np.float32)
sample_count = np.asarray(live_field_payload["sample_count"], dtype=np.float32)
observed_mask = np.asarray(live_field_payload["observed_mask"], dtype=bool)
unsupported_mask = ~observed_mask
support_fraction = (sample_count / max(float(np.nanmax(sample_count)), float(lfq.EPS))).astype(np.float32)
blend_weight = np.clip(support_fraction, 0.0, 1.0).astype(np.float32)

field_low = lfq.smooth_masked_image(
    masked_mean=masked_mean,
    observed_mask=observed_mask.astype(np.float32),
    sigma_px=LIVE_FIELD_SIGMA_LOW_PX,
    mode=LIVE_FIELD_MODE,
).astype(np.float32)
field_high = lfq.smooth_masked_image(
    masked_mean=masked_mean,
    observed_mask=observed_mask.astype(np.float32),
    sigma_px=LIVE_FIELD_SIGMA_HIGH_PX,
    mode=LIVE_FIELD_MODE,
).astype(np.float32)

field_low = (field_low / np.median(field_low[np.isfinite(field_low) & observed_mask])).astype(np.float32)
field_high = (field_high / np.median(field_high[np.isfinite(field_high) & observed_mask])).astype(np.float32)
live_field = (blend_weight * field_low + (1.0 - blend_weight) * field_high).astype(np.float32)
live_field = (live_field / np.median(live_field[np.isfinite(live_field) & observed_mask])).astype(np.float32)

used_mask = live_field_position_df["used_for_field"].astype(bool)
used_background_pixels = int(np.nansum(live_field_position_df.loc[used_mask, "background_pixels"].astype(float))) if len(live_field_position_df) else 0
live_field_summary_df = pd.DataFrame(
    [
        {
            "channel": "live_tagyfp",
            "used_positions": int(np.sum(used_mask)),
            "used_background_pixels": used_background_pixels,
            "field_min": float(np.nanmin(live_field)),
            "field_p01": float(np.nanpercentile(live_field, 1.0)),
            "field_median": float(np.nanmedian(live_field)),
            "field_p99": float(np.nanpercentile(live_field, 99.0)),
            "field_max": float(np.nanmax(live_field)),
            "field_center_value": float(live_field[live_field.shape[0] // 2, live_field.shape[1] // 2]),
            "sample_coverage_fraction": float(np.mean(observed_mask)),
            "field_sigma_low_px": float(LIVE_FIELD_SIGMA_LOW_PX),
            "field_sigma_high_px": float(LIVE_FIELD_SIGMA_HIGH_PX),
            "field_blend_rule": str(LIVE_FIELD_BLEND_RULE),
            "min_background_pixels": int(LIVE_FIELD_MIN_BACKGROUND_PIXELS),
            "smoothing_mode": str(LIVE_FIELD_MODE),
        }
    ]
)
live_field_summary_df.to_csv(OUT_DIR / "live_yfp_flatfield_summary.tsv", sep="\t", index=False)
live_field_position_df.to_csv(OUT_DIR / "live_yfp_flatfield_position_diagnostics.tsv", sep="\t", index=False)
np.savez_compressed(
    OUT_DIR / "live_yfp_flatfield_field.npz",
    field=live_field.astype(np.float32),
    field_low=field_low.astype(np.float32),
    field_high=field_high.astype(np.float32),
    sample_count=sample_count.astype(np.float32),
    observed_mask=observed_mask.astype(np.uint8),
    masked_mean=masked_mean.astype(np.float32),
    support_fraction=support_fraction.astype(np.float32),
    blend_weight=blend_weight.astype(np.float32),
)

print("Live flat-field summary:")
display(live_field_summary_df)
print()
print("Per-position off-mask support summary:")
display(live_field_position_df.head())

projected_only = np.where(unsupported_mask, live_field, np.nan).astype(np.float32)
field_stack = np.concatenate([field_low.ravel(), field_high.ravel(), live_field.ravel()])
field_stack = field_stack[np.isfinite(field_stack)]
field_vmin = float(np.nanpercentile(field_stack, 1.0))
field_vmax = float(np.nanpercentile(field_stack, 99.0))

fig, axes = plt.subplots(2, 4, figsize=(24.0, 10.0), constrained_layout=True)
axes = np.asarray(axes)

axes[0, 0].imshow(robust_rescale_from_values(masked_mean, observed_mask), cmap="Greens")
axes[0, 0].set_title("Global normalized off-mask mean\n(observed pixels only)")

im = axes[0, 1].imshow(support_fraction, cmap="viridis", vmin=0.0, vmax=1.0)
axes[0, 1].set_title("Global support fraction\n(sample count / max)")
fig.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

im = axes[0, 2].imshow(blend_weight, cmap="viridis", vmin=0.0, vmax=1.0)
axes[0, 2].set_title("Blend weight for low-sigma field\n1 = trust low-sigma, 0 = trust high-sigma")
fig.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

axes[0, 3].imshow(unsupported_mask.astype(float), cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 3].set_title("Global unsupported region\n(no direct background support)")

im = axes[1, 0].imshow(field_low, cmap="magma", vmin=field_vmin, vmax=field_vmax)
lfq._mask_contour(axes[1, 0], unsupported_mask, color="white")
axes[1, 0].set_title(f"Global low-sigma field\nsigma = {LIVE_FIELD_SIGMA_LOW_PX:.0f} px")
fig.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

im = axes[1, 1].imshow(field_high, cmap="magma", vmin=field_vmin, vmax=field_vmax)
lfq._mask_contour(axes[1, 1], unsupported_mask, color="white")
axes[1, 1].set_title(f"Global high-sigma field\nsigma = {LIVE_FIELD_SIGMA_HIGH_PX:.0f} px")
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

im = axes[1, 2].imshow(projected_only, cmap="magma", vmin=field_vmin, vmax=field_vmax)
axes[1, 2].set_title("Final blended field\nunsupported region only")
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

im = axes[1, 3].imshow(live_field, cmap="magma", vmin=field_vmin, vmax=field_vmax)
lfq._mask_contour(axes[1, 3], unsupported_mask, color="white")
axes[1, 3].set_title("Final blended live YFP flat field\nwhite = unsupported region")
fig.colorbar(im, ax=axes[1, 3], fraction=0.046, pad=0.04)
axes[1, 3].text(
    0.02,
    0.02,
    f"used positions = {int(live_field_summary_df['used_positions'].iloc[0])}\n"
    f"coverage = {float(live_field_summary_df['sample_coverage_fraction'].iloc[0]):.3f}\n"
    f"low/high sigma = {float(live_field_summary_df['field_sigma_low_px'].iloc[0]):.0f}/{float(live_field_summary_df['field_sigma_high_px'].iloc[0]):.0f} px\n"
    f"blend = {str(live_field_summary_df['field_blend_rule'].iloc[0])}\n"
    f"p01-p99 = {float(live_field_summary_df['field_p01'].iloc[0]):.3f} to {float(live_field_summary_df['field_p99'].iloc[0]):.3f}",
    transform=axes[1, 3].transAxes,
    ha="left",
    va="bottom",
    fontsize=8,
    bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9},
)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(QC_DIR / "live_yfp_flatfield_estimate.png", dpi=160, bbox_inches="tight")
plt.show()


## Apply Flat Field And Fit Live Background

In [ ]:

ok_positions = live_mask_summary_df.loc[live_mask_summary_df["status"] == "ok", "canonical_position"].astype(str).tolist()

all_corrected_values = []
offmask_corrected_values = []
flatfield_diag_rows = []

for pos in ok_positions:
    live_ctx = load_live_bundle(pos)
    live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
    raw = live_ctx["yfp_raw"].astype(np.float32)
    corrected = lfq.apply_illumination_field(raw, live_field).astype(np.float32)
    valid = np.isfinite(corrected) & (raw > 0)
    offmask = valid & (~live_mask)
    all_corrected_values.append(corrected[valid].astype(np.float32))
    if np.any(offmask):
        offmask_corrected_values.append(corrected[offmask].astype(np.float32))
    flatfield_diag_rows.append(
        {
            "canonical_position": pos,
            "raw_offmask_median": float(np.median(raw[offmask])) if np.any(offmask) else np.nan,
            "corrected_offmask_median": float(np.median(corrected[offmask])) if np.any(offmask) else np.nan,
            "raw_offmask_p10": float(np.percentile(raw[offmask], 10.0)) if np.any(offmask) else np.nan,
            "corrected_offmask_p10": float(np.percentile(corrected[offmask], 10.0)) if np.any(offmask) else np.nan,
            "raw_offmask_p90": float(np.percentile(raw[offmask], 90.0)) if np.any(offmask) else np.nan,
            "corrected_offmask_p90": float(np.percentile(corrected[offmask], 90.0)) if np.any(offmask) else np.nan,
            "offmask_pixels": int(np.sum(offmask)),
        }
    )

all_corrected_values = np.concatenate(all_corrected_values).astype(np.float32)
offmask_corrected_values = np.concatenate(offmask_corrected_values).astype(np.float32)
flatfield_diagnostics_df = pd.DataFrame(flatfield_diag_rows).sort_values("canonical_position").reset_index(drop=True)
flatfield_diagnostics_df.to_csv(OUT_DIR / "live_yfp_flatfield_background_diagnostics.tsv", sep="	", index=False)

live_bg_fit = lfq.fit_background_from_image_pixels(
    image=offmask_corrected_values,
    n_bins=16384,
    smooth_sigma_bins=6.0,
    exclude_exact_zero_pixels=True,
)
live_bg_fit["n_images"] = int(len(ok_positions))
live_bg_fit["n_pixels"] = int(offmask_corrected_values.size)
live_bg_fit["fit_method"] = str(live_bg_fit["fit_method"]) + ";off_mask_pixels_after_flatfield"

live_bg_df = pd.DataFrame(
    [
        {
            "channel": "live_tagyfp_flatfield_corrected",
            "mu_bg_raw": float(live_bg_fit["mu_bg_raw"]),
            "sigma_bg_raw": float(live_bg_fit["sigma_bg_raw"]),
            "mode_idx": int(live_bg_fit["mode_idx"]),
            "n_images": int(live_bg_fit["n_images"]),
            "n_pixels": int(live_bg_fit["n_pixels"]),
            "global_min": float(live_bg_fit["global_min"]),
            "global_max": float(live_bg_fit["global_max"]),
            "fit_method": str(live_bg_fit["fit_method"]),
        }
    ]
)
live_bg_df.to_csv(OUT_DIR / "global_live_background_params.tsv", sep="	", index=False)

print("Active global off-mask background model used downstream:")
display(live_bg_df)
print()
print("Off-mask median stabilization after flat field:")
display(flatfield_diagnostics_df.head())

per_image_bg_rows = []
for pos in ok_positions:
    live_ctx = load_live_bundle(pos)
    live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
    raw = live_ctx["yfp_raw"].astype(np.float32)
    corrected = lfq.apply_illumination_field(raw, live_field).astype(np.float32)
    valid = np.isfinite(corrected) & (raw > 0)
    offmask = valid & (~live_mask)
    if not np.any(offmask):
        per_image_bg_rows.append(
            {
                "canonical_position": pos,
                "offmask_pixels": 0,
                "mu_bg_raw": np.nan,
                "sigma_bg_raw": np.nan,
                "corrected_offmask_median": np.nan,
                "corrected_offmask_p10": np.nan,
                "corrected_offmask_p90": np.nan,
                "offmask_centered_median": np.nan,
                "offmask_centered_p10": np.nan,
                "offmask_centered_p90": np.nan,
                "inside_positive_pixels_after_per_image_subtraction": 0,
            }
        )
        continue
    fit_i = lfq.fit_background_from_image_pixels(
        image=corrected[offmask].astype(np.float32),
        n_bins=8192,
        smooth_sigma_bins=6.0,
        exclude_exact_zero_pixels=True,
    )
    mu_i = float(fit_i["mu_bg_raw"])
    sigma_i = float(fit_i["sigma_bg_raw"])
    centered_offmask = corrected[offmask].astype(np.float32) - mu_i
    bgsub_i = common.subtract_uniform_background(
        image=corrected,
        mu_bg_raw=mu_i,
        clip_below_zero=True,
    ).astype(np.float32)
    keep_inside = live_mask & np.isfinite(bgsub_i) & (bgsub_i > 0)
    per_image_bg_rows.append(
        {
            "canonical_position": pos,
            "offmask_pixels": int(np.sum(offmask)),
            "mu_bg_raw": mu_i,
            "sigma_bg_raw": sigma_i,
            "corrected_offmask_median": float(np.median(corrected[offmask])),
            "corrected_offmask_p10": float(np.percentile(corrected[offmask], 10.0)),
            "corrected_offmask_p90": float(np.percentile(corrected[offmask], 90.0)),
            "offmask_centered_median": float(np.median(centered_offmask)),
            "offmask_centered_p10": float(np.percentile(centered_offmask, 10.0)),
            "offmask_centered_p90": float(np.percentile(centered_offmask, 90.0)),
            "inside_positive_pixels_after_per_image_subtraction": int(np.sum(keep_inside)),
        }
    )

per_image_bg_df = pd.DataFrame(per_image_bg_rows).sort_values("canonical_position").reset_index(drop=True)
per_image_bg_df.to_csv(OUT_DIR / "live_yfp_per_image_background_params.tsv", sep="	", index=False)

print("Per-image off-mask background fits after flat-field correction:")
display(per_image_bg_df.head())


def select_parameter_outlier_positions(df, column):
    vals = df[column].to_numpy(dtype=float)
    q1, q3 = np.percentile(vals, [25.0, 75.0])
    iqr = q3 - q1
    low_cut = q1 - 1.5 * iqr
    high_cut = q3 + 1.5 * iqr
    low_df = df[df[column] < low_cut].sort_values(column, ascending=True)
    high_df = df[df[column] > high_cut].sort_values(column, ascending=False)
    if low_df.empty:
        low_df = df.nsmallest(1, column)
        low_reason = f"lowest {column} (no Tukey low outlier)"
    else:
        low_reason = f"low {column} outlier"
    if high_df.empty:
        high_df = df.nlargest(1, column)
        high_reason = f"highest {column} (no Tukey high outlier)"
    else:
        high_reason = f"high {column} outlier"
    return [
        (str(low_df.iloc[0]["canonical_position"]), low_reason),
        (str(high_df.iloc[0]["canonical_position"]), high_reason),
    ]


def build_per_image_background_render_payloads(example_positions):
    payloads = []
    raw_corrected_vals_local = []
    full_bgsub_vals_local = []
    inside_positive_vals_local = []
    for pos in example_positions:
        live_ctx = load_live_bundle(pos)
        live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
        raw = live_ctx["yfp_raw"].astype(np.float32)
        corrected = lfq.apply_illumination_field(raw, live_field).astype(np.float32)
        fit_i = lfq.fit_background_from_image_pixels(
            image=corrected[(np.isfinite(corrected) & (raw > 0) & (~live_mask))].astype(np.float32),
            n_bins=8192,
            smooth_sigma_bins=6.0,
            exclude_exact_zero_pixels=True,
        )
        mu_i = float(fit_i["mu_bg_raw"])
        sigma_i = float(fit_i["sigma_bg_raw"])
        bgsub_i = common.subtract_uniform_background(
            image=corrected,
            mu_bg_raw=mu_i,
            clip_below_zero=True,
        ).astype(np.float32)
        raw_corrected_vals_local.append(raw[np.isfinite(raw) & (raw > 0)].astype(np.float32))
        raw_corrected_vals_local.append(corrected[np.isfinite(corrected) & (corrected > 0)].astype(np.float32))
        full_bgsub_vals_local.append(bgsub_i[np.isfinite(bgsub_i)].astype(np.float32))
        keep_inside_rep = live_mask & np.isfinite(bgsub_i) & (bgsub_i > 0)
        if np.any(keep_inside_rep):
            inside_positive_vals_local.append(bgsub_i[keep_inside_rep].astype(np.float32))
        payloads.append(
            {
                "canonical_position": pos,
                "live_ctx": live_ctx,
                "live_mask": live_mask,
                "raw": raw,
                "corrected": corrected,
                "bgsub": bgsub_i,
                "fit": fit_i,
                "mu_i": mu_i,
                "sigma_i": sigma_i,
            }
        )

    raw_corrected_vals_local = np.concatenate(raw_corrected_vals_local).astype(np.float32) if raw_corrected_vals_local else np.array([], dtype=np.float32)
    full_bgsub_vals_local = np.concatenate(full_bgsub_vals_local).astype(np.float32) if full_bgsub_vals_local else np.array([], dtype=np.float32)
    inside_positive_vals_local = np.concatenate(inside_positive_vals_local).astype(np.float32) if inside_positive_vals_local else np.array([], dtype=np.float32)

    raw_vmin_local = float(np.percentile(raw_corrected_vals_local, 1.0)) if raw_corrected_vals_local.size else 0.0
    raw_vmax_local = float(np.percentile(raw_corrected_vals_local, 99.0)) if raw_corrected_vals_local.size else 1.0
    bgsub_vmin_local = 0.0
    bgsub_vmax_local = float(np.percentile(full_bgsub_vals_local, 99.5)) if full_bgsub_vals_local.size else 1.0
    proc_shared_xlim_local = (0.0, float(np.nanpercentile(inside_positive_vals_local, 99.5))) if inside_positive_vals_local.size else (0.0, 1.0)
    return {
        "payloads": payloads,
        "raw_vmin": raw_vmin_local,
        "raw_vmax": raw_vmax_local,
        "bgsub_vmin": bgsub_vmin_local,
        "bgsub_vmax": bgsub_vmax_local,
        "proc_shared_xlim": proc_shared_xlim_local,
    }


def render_per_image_background_examples(example_positions, label_prefix, filename_prefix, shared_render_limits=None):
    render_payload = build_per_image_background_render_payloads(example_positions)
    if shared_render_limits is None:
        raw_vmin_local = float(render_payload["raw_vmin"])
        raw_vmax_local = float(render_payload["raw_vmax"])
        bgsub_vmin_local = float(render_payload["bgsub_vmin"])
        bgsub_vmax_local = float(render_payload["bgsub_vmax"])
        proc_shared_xlim_local = tuple(render_payload["proc_shared_xlim"])
    else:
        raw_vmin_local = float(shared_render_limits["raw_vmin"])
        raw_vmax_local = float(shared_render_limits["raw_vmax"])
        bgsub_vmin_local = float(shared_render_limits["bgsub_vmin"])
        bgsub_vmax_local = float(shared_render_limits["bgsub_vmax"])
        proc_shared_xlim_local = tuple(shared_render_limits["proc_shared_xlim"])

    for payload in render_payload["payloads"]:
        pos = payload["canonical_position"]
        live_ctx = payload["live_ctx"]
        live_mask = np.asarray(payload["live_mask"], dtype=bool)
        raw = np.asarray(payload["raw"], dtype=np.float32)
        corrected = np.asarray(payload["corrected"], dtype=np.float32)
        bgsub_i = np.asarray(payload["bgsub"], dtype=np.float32)
        cent_sub = live_cent_df[
            (live_cent_df["canonical_position"].astype(str) == str(pos))
            & (live_cent_df["mapping_status"] == "ok")
            & (live_cent_df["annotation_status"] == "annotated")
            & (live_cent_df["inside_small_fov"].fillna(False).astype(bool))
        ].copy()
        beads_xy = cent_sub[["centroid_small_x_px", "centroid_small_y_px"]].to_numpy(dtype=np.float32) if len(cent_sub) else np.zeros((0, 2), dtype=np.float32)
        fit_i = payload["fit"]
        mu_i = float(payload["mu_i"])
        sigma_i = float(payload["sigma_i"])
        valid = np.isfinite(corrected) & (raw > 0)
        offmask = valid & (~live_mask)

        corrected_hist = lfq.background_histogram_from_pixels(
            image=corrected[valid].astype(np.float32),
            n_bins=1024,
            exclude_exact_zero_pixels=True,
        )
        fit_x = np.asarray(fit_i["centers"], dtype=float)
        fit_counts = np.asarray(fit_i["counts"], dtype=float)
        fit_edges = np.asarray(fit_i["edges"], dtype=float)
        fit_widths = np.diff(fit_edges)
        fit_smooth = np.asarray(fit_i["smooth_counts"], dtype=float)
        fit_curve = np.asarray(fit_i["fit_counts"], dtype=float)
        keep_inside = live_mask & np.isfinite(bgsub_i) & (bgsub_i > 0)
        if np.any(keep_inside):
            processed_hist_i = lfq.background_histogram_from_pixels(
                image=bgsub_i[keep_inside].astype(np.float32),
                n_bins=1024,
                exclude_exact_zero_pixels=False,
            )
            proc_x = np.asarray(processed_hist_i["centers"], dtype=float)
            proc_counts = np.asarray(processed_hist_i["counts"], dtype=float)
            proc_edges = np.asarray(processed_hist_i["edges"], dtype=float)
            proc_widths = np.diff(proc_edges)
        else:
            proc_x = np.array([], dtype=float)
            proc_counts = np.array([], dtype=float)
            proc_widths = np.array([], dtype=float)

        full_x = np.asarray(corrected_hist["centers"], dtype=float)
        full_counts = np.asarray(corrected_hist["counts"], dtype=float)
        full_edges = np.asarray(corrected_hist["edges"], dtype=float)
        full_widths = np.diff(full_edges)
        fit_xlim = (float(fit_i["global_min"]), min(float(fit_i["global_max"]), mu_i + 6.0 * sigma_i))

        print(f"{label_prefix} scene {pos} | per-image off-mask fit")
        fig, axes = plt.subplots(1, 3, figsize=(18.0, 4.0), constrained_layout=True)

        ax = axes[0]
        ax.bar(full_x, full_counts, width=full_widths, align="center", color="0.85", edgecolor="0.70", linewidth=0.2)
        ax.set_title(f"{pos} flat-field corrected\nfull pixel distribution")
        ax.set_xlabel("Flat-field corrected intensity")
        ax.set_ylabel("Pixel count")

        ax = axes[1]
        ax.bar(fit_x, fit_counts, width=fit_widths, align="center", color="0.85", edgecolor="0.70", linewidth=0.2, label="off-mask histogram")
        ax.plot(fit_x, fit_smooth, color="k", lw=1.5, label="smoothed histogram")
        ax.plot(fit_x, fit_curve, color="tab:blue", lw=1.8, label="left-half Gaussian")
        ax.axvline(mu_i, color="tab:red", ls="--", lw=1.4, label="mu_bg")
        ax.axvline(mu_i - sigma_i, color="tab:red", ls=":", lw=1.1, alpha=0.9, label="mu_bg ± sigma_bg")
        ax.axvline(mu_i + sigma_i, color="tab:red", ls=":", lw=1.1, alpha=0.9)
        ax.set_xlim(*fit_xlim)
        ax.set_title(f"{pos} off-mask null/background view\nper-image fit after flat-field correction")
        ax.set_xlabel("Flat-field corrected intensity")
        ax.set_ylabel("Pixel count")
        ax.text(
            0.98,
            0.98,
            f"off-mask px={int(np.sum(offmask))}\nmu_bg={mu_i:.1f}\nsigma_bg={sigma_i:.1f}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8,
            bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9},
        )
        ax.legend(frameon=False, fontsize=8, loc="lower left")

        ax = axes[2]
        if proc_x.size:
            ax.bar(proc_x, proc_counts, width=proc_widths, align="center", color="0.85", edgecolor="0.70", linewidth=0.2)
            ax.set_xlim(*proc_shared_xlim_local)
        ax.set_title(f"{pos} per-image bg-subtracted\ninside transferred DAPI mask (>0 only)")
        ax.set_xlabel("Per-image corrected + bg-subtracted intensity")
        ax.set_ylabel("Pixel count")

        fig.savefig(QC_DIR / f"{pos}_{filename_prefix}_hist.png", dpi=160, bbox_inches="tight")
        plt.show()

        print(f"{label_prefix} scene {pos} | images after per-image subtraction")
        fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.0), constrained_layout=True)
        axes = np.atleast_1d(axes)
        axes[0].imshow(common._robust_rescale(live_ctx["bf"]), cmap="gray")
        lfq._mask_contour(axes[0], live_mask, color="cyan")
        axes[0].set_title(f"{pos} BF + transferred mask\nmagenta circles = bead-containing wells")

        axes[1].imshow(raw, cmap="Greens", vmin=raw_vmin_local, vmax=raw_vmax_local)
        lfq._mask_contour(axes[1], live_mask, color="cyan")
        axes[1].set_title("TagYFP raw\nshared representative scale")

        axes[2].imshow(corrected, cmap="Greens", vmin=raw_vmin_local, vmax=raw_vmax_local)
        lfq._mask_contour(axes[2], live_mask, color="cyan")
        axes[2].set_title("Flat-field corrected\nshared representative scale")

        axes[3].imshow(bgsub_i, cmap="Greens", vmin=bgsub_vmin_local, vmax=bgsub_vmax_local)
        lfq._mask_contour(axes[3], live_mask, color="cyan")
        axes[3].set_title("Per-image bg-subtracted\nfull image, shared representative scale")

        if beads_xy.size:
            for ax in axes:
                ax.scatter(
                    beads_xy[:, 0],
                    beads_xy[:, 1],
                    s=42,
                    facecolors="none",
                    edgecolors="magenta",
                    linewidths=1.2,
                    label="bead-containing wells in small FOV",
                )

        if beads_xy.size:
            handles, labels = axes[0].get_legend_handles_labels()
            if labels:
                axes[0].legend(handles[:1], labels[:1], frameon=False, fontsize=7, loc="lower left")

        for ax in axes:
            ax.set_xticks([])
            ax.set_yticks([])

        fig.savefig(QC_DIR / f"{pos}_{filename_prefix}_images.png", dpi=160, bbox_inches="tight")
        plt.show()


per_image_example_positions = [p for p in FLATFIELD_EXAMPLE_POSITIONS if p in ok_positions]
if not per_image_example_positions:
    per_image_example_positions = ok_positions[:5]

summary_df_for_outliers = per_image_bg_df[np.isfinite(per_image_bg_df["mu_bg_raw"]) & np.isfinite(per_image_bg_df["sigma_bg_raw"])].copy()
outlier_records_for_limits = []
for metric in ["mu_bg_raw", "sigma_bg_raw"]:
    outlier_records_for_limits.extend(select_parameter_outlier_positions(summary_df_for_outliers, metric))
outlier_positions_for_limits = []
for pos, _ in outlier_records_for_limits:
    if pos not in outlier_positions_for_limits:
        outlier_positions_for_limits.append(pos)
combined_render_positions = []
for pos in per_image_example_positions + outlier_positions_for_limits:
    if pos not in combined_render_positions:
        combined_render_positions.append(pos)
combined_render_limits = build_per_image_background_render_payloads(combined_render_positions)

render_per_image_background_examples(
    example_positions=per_image_example_positions,
    label_prefix="Representative",
    filename_prefix="live_yfp_per_image_background",
    shared_render_limits=combined_render_limits,
)


## Per-Image Background Parameter Summary

These plots summarize the **per-image off-mask** `mu_bg` and `sigma_bg` fits after flat-field correction. `mu_bg` can shift image to image; `sigma_bg` is expected to be more stable if the residual background noise properties are similar across scenes.


In [ ]:
summary_df = per_image_bg_df[np.isfinite(per_image_bg_df["mu_bg_raw"]) & np.isfinite(per_image_bg_df["sigma_bg_raw"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2), constrained_layout=True)

mu_vals = summary_df["mu_bg_raw"].to_numpy(dtype=float)
sigma_vals = summary_df["sigma_bg_raw"].to_numpy(dtype=float)

axes[0].hist(mu_vals, bins=16, color="0.85", edgecolor="0.55")
axes[0].axvline(float(np.median(mu_vals)), color="tab:red", ls="--", lw=1.5, label="median")
axes[0].set_title("Per-image off-mask mu_bg distribution")
axes[0].set_xlabel("mu_bg (flat-field corrected intensity units)")
axes[0].set_ylabel("Image count")
axes[0].text(
    0.98,
    0.98,
    f"n={len(mu_vals)}\nmedian={float(np.median(mu_vals)):.1f}\nIQR={float(np.percentile(mu_vals,75)-np.percentile(mu_vals,25)):.1f}",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=8,
    bbox={"facecolor":"white","edgecolor":"0.85","boxstyle":"round,pad=0.25","alpha":0.9},
)
axes[0].legend(frameon=False, fontsize=8, loc="upper left")

axes[1].hist(sigma_vals, bins=16, color="0.85", edgecolor="0.55")
axes[1].axvline(float(np.median(sigma_vals)), color="tab:red", ls="--", lw=1.5, label="median")
axes[1].set_title("Per-image off-mask sigma_bg distribution")
axes[1].set_xlabel("sigma_bg (flat-field corrected intensity units)")
axes[1].set_ylabel("Image count")
axes[1].text(
    0.98,
    0.98,
    f"n={len(sigma_vals)}\nmedian={float(np.median(sigma_vals)):.1f}\nIQR={float(np.percentile(sigma_vals,75)-np.percentile(sigma_vals,25)):.1f}",
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    fontsize=8,
    bbox={"facecolor":"white","edgecolor":"0.85","boxstyle":"round,pad=0.25","alpha":0.9},
)
axes[1].legend(frameon=False, fontsize=8, loc="upper left")

fig.savefig(QC_DIR / "live_yfp_per_image_background_summary.png", dpi=160, bbox_inches="tight")
plt.show()


## Per-Image Background Parameter Outliers

Show the low- and high-end outlier scenes for `mu_bg` and `sigma_bg` using the exact same histogram-and-image diagnostics as above, now organized around parameter outliers.

In [ ]:

summary_df = per_image_bg_df[np.isfinite(per_image_bg_df["mu_bg_raw"]) & np.isfinite(per_image_bg_df["sigma_bg_raw"])].copy()

outlier_records = []
for metric in ["mu_bg_raw", "sigma_bg_raw"]:
    outlier_records.extend(select_parameter_outlier_positions(summary_df, metric))

outlier_reason_map = {}
for pos, reason in outlier_records:
    outlier_reason_map.setdefault(pos, []).append(reason)

outlier_positions = list(outlier_reason_map.keys())
outlier_df = pd.DataFrame(
    [
        {
            "canonical_position": pos,
            "reasons": "; ".join(outlier_reason_map[pos]),
            "mu_bg_raw": float(summary_df.loc[summary_df["canonical_position"] == pos, "mu_bg_raw"].iloc[0]),
            "sigma_bg_raw": float(summary_df.loc[summary_df["canonical_position"] == pos, "sigma_bg_raw"].iloc[0]),
        }
        for pos in outlier_positions
    ]
)

print("Per-image background parameter outliers / extremes shown below:")
display(outlier_df)
print()
for pos in outlier_positions:
    print(f"{pos}: {'; '.join(outlier_reason_map[pos])}")
print()

render_per_image_background_examples(
    example_positions=outlier_positions,
    label_prefix="Outlier",
    filename_prefix="live_yfp_per_image_background_outlier",
    shared_render_limits=combined_render_limits,
)


## Downstream Live Plotting

Downstream live distance-trace quantification and plotting now live in `06_live_plotting.ipynb`, which loads the saved masks, flat-field estimate, and per-image background fits produced here.